# Week 2 Goal

Compare different structured extraction strategies and measure reliability:

- Extraction Strategies
- Naive JSON prompt
- Schema-guided prompt
- Strict + validation-aware prompt



---



## Install Dependencies

In [1]:
# Import pydantic
!pip install openai pydantic python-dotenv

In [2]:
#Import anthropic
!pip install anthropic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 662.1/662.1 kB 7.6 MB/s eta 0:00:00


## Libraries

In [3]:
import anthropic
from google.colab import userdata

client = anthropic.Anthropic(
    api_key=userdata.get('ANTHROPIC_API_KEY')
)

## Model Restrictions

- For budget limiting tokens to 1,000
- Good enough for structured extraction
- No randomness, Temp = 0

In [4]:
# Set Model
model="claude-sonnet-4-6"

In [5]:
# Call LLM with 1000 token limit, temp=0 for consistency
def call_llm(prompt, model="claude-sonnet-4-6"):
    import time

    start = time.time()

    response = client.messages.create(
        model=model,
        max_tokens=1000,
        temperature=0,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )

    latency = time.time() - start
    output = response.content[0].text

    return {
        "output": output,
        "latency": latency,
        "input_tokens": response.usage.input_tokens,
        "output_tokens": response.usage.output_tokens
    }

In [6]:
# Test Model
call_llm("Return JSON: {hello: world}")

{'output': '```json\n{\n  "hello": "world"\n}\n```',
 'latency': 1.190303087234497,
 'input_tokens': 15,
 'output_tokens': 19}

**Insight**

- Passes formatting requirement (valid JSON)
- No hallucinations or malformed structure
- No meaningful task completion beyond echoing input
- No reasoning, extraction, or transformation demonstrated



---



## Use Case: Fintech Customer Support Ticket Intake

- This mirrors a production triage system where an LLM extracts structured risk signals from messy user messages.

In [7]:
# Input Examples to Test
examples = [
"""
Hi, I noticed two charges for $842 from merchants I do not recognize.
Card ending 4412 may be compromised. I am currently traveling in Mexico.
Please freeze my card and dispute both transactions.
""",

"""
I missed my loan payment due March 1 because I lost my job.
Can I request hardship assistance and avoid collections?
""",

"""
I received an SMS saying my account is locked and asking for my password.
Is this fraud? The message came from +1 800 555 1212.
"""
]

## Target Schema

In [8]:
# Target Schema
from pydantic import BaseModel
from typing import List, Optional

class SupportExtraction(BaseModel):
    issue_type: str
    urgency_level: str
    fraud_flag: bool
    monetary_amount: Optional[float]
    requested_actions: List[str]
    sentiment: str



---



## 3 Method Approach

**Method 1: Naive Prompting (Baseline)**
- Represents how many teams start in production: simple instructions asking the model to return JSON. It establishes a baseline for comparison and reveals common failures such as malformed JSON, missing fields, or hallucinated values.

**Method 2: Schema Guided Prompting (Structure Control)**
- Adds an explicit schema to test whether stronger formatting constraints improve compliance and field level accuracy.

**Method 3: Strict Validation Aware Prompting (Production Hardening)**
- Introduces business rules such as allowed labels, null handling, and “never invent values,” which simulates guardrails used in production systems.

In [9]:
# Evaluation — 3 methods, each prompt built per-ticket (text not examples)
import json
import pandas as pd

results = []

for text in examples:
    for method, method_prompt in {

        "naive": f"""
Extract as JSON:

Fields:
issue_type
urgency_level
fraud_flag
monetary_amount
requested_actions
sentiment

Text:
{text}
""",

        "schema": f"""
Return only valid JSON matching this schema:

{{
  "issue_type": string,
  "urgency_level": string,
  "fraud_flag": boolean,
  "monetary_amount": number or null,
  "requested_actions": [string],
  "sentiment": string
}}

Input:
{text}
""",

        "strict": f"""
Return valid JSON only. No explanation, no markdown, no code blocks.

Rules:
- Use null if a value is missing
- Never invent values
- requested_actions must be a list
- urgency_level must be exactly: low, medium, or high
- issue_type must be exactly one of: fraud, hardship, phishing, dispute

Input:
{text}
"""

    }.items():

        r = call_llm(method_prompt)  # ✅ method_prompt, not prompt

        try:
            parsed = json.loads(r["output"])
            valid_json = True
        except:
            valid_json = False
            parsed = {}

        results.append({
            "method": method,
            "valid_json": valid_json,
            "latency": r["latency"],
            "input_tokens": r["input_tokens"],
            "output_tokens": r["output_tokens"]
        })

df = pd.DataFrame(results)
print(df.groupby("method")[["valid_json", "latency", "input_tokens", "output_tokens"]].mean())

        valid_json   latency  input_tokens  output_tokens
method                                                   
naive          0.0  3.486748     78.666667     169.000000
schema         0.0  1.860066    115.666667      88.000000
strict         1.0  2.421404    126.666667      84.666667


**Insight**

- Schema is fastest (1.86s) but produces the least reliable output — speed means nothing if the JSON is broken

- Strict uses the most input tokens (126 avg) because the rules add prompt length, but produces the fewest output tokens (84) — the model is being more disciplined and concise

- Naive outputs the most tokens (169) — the model is probably rambling or wrapping in markdown since it has no constraints




---



##

## Field-level accuracy

Did the model extract the right values, not just valid JSON?

In [10]:
# Ground Truth Labels — expected values per ticket
ground_truth = [
    {
        "issue_type": "fraud",
        "urgency_level": "high",
        "fraud_flag": True,
        "monetary_amount": 842.0,
        "requested_actions": ["freeze card", "dispute transactions"],
        "sentiment": "alarmed"
    },
    {
        "issue_type": "hardship",
        "urgency_level": "medium",
        "fraud_flag": False,
        "monetary_amount": None,
        "requested_actions": ["hardship assistance", "avoid collections"],
        "sentiment": "distressed"
    },
    {
        "issue_type": "phishing",
        "urgency_level": "high",
        "fraud_flag": True,
        "monetary_amount": None,
        "requested_actions": ["verify message legitimacy"],
        "sentiment": "concerned"
    }
]

In [11]:
# Field-level scorer
def score_extraction(parsed, truth):
    scores = {}

    # Exact match fields
    for field in ["issue_type", "urgency_level", "fraud_flag", "monetary_amount"]:
        scores[field] = int(parsed.get(field) == truth[field])

    # List field — score by overlap (how many expected actions were captured)
    predicted_actions = [a.lower() for a in parsed.get("requested_actions", [])]
    expected_actions  = [a.lower() for a in truth["requested_actions"]]
    if expected_actions:
        matches = sum(
            any(exp in pred or pred in exp for pred in predicted_actions)
            for exp in expected_actions
        )
        scores["requested_actions"] = matches / len(expected_actions)
    else:
        scores["requested_actions"] = 1.0

    # Sentiment — exact match (flexible, no fixed labels)
    scores["sentiment"] = int(parsed.get("sentiment") == truth["sentiment"])

    scores["overall"] = sum(scores.values()) / len(scores)
    return scores

In [12]:
# Full evaluation with field-level scoring
import json
import pandas as pd

results = []
ticket_index = 0

for text, truth in zip(examples, ground_truth):
    ticket_index += 1

    for method, method_prompt in {

        "naive": f"""
Extract as JSON:

Fields:
issue_type
urgency_level
fraud_flag
monetary_amount
requested_actions
sentiment

Text:
{text}
""",

        "schema": f"""
Return only valid JSON matching this schema:

{{
  "issue_type": string,
  "urgency_level": string,
  "fraud_flag": boolean,
  "monetary_amount": number or null,
  "requested_actions": [string],
  "sentiment": string
}}

Input:
{text}
""",

        "strict": f"""
Return valid JSON only. No explanation, no markdown, no code blocks.

Rules:
- Use null if a value is missing
- Never invent values
- requested_actions must be a list
- urgency_level must be exactly: low, medium, or high
- issue_type must be exactly one of: fraud, hardship, phishing, dispute

Input:
{text}
"""

    }.items():

        r = call_llm(method_prompt)

        try:
            parsed = json.loads(r["output"])
            valid_json = True
        except:
            valid_json = False
            parsed = {}

        field_scores = score_extraction(parsed, truth) if valid_json else {
            "issue_type": 0, "urgency_level": 0, "fraud_flag": 0,
            "monetary_amount": 0, "requested_actions": 0,
            "sentiment": 0, "overall": 0
        }

        results.append({
            "ticket": ticket_index,
            "method": method,
            "valid_json": valid_json,
            "latency": r["latency"],
            "input_tokens": r["input_tokens"],
            "output_tokens": r["output_tokens"],
            **field_scores
        })

df = pd.DataFrame(results)

print("=== Field-Level Accuracy by Method ===\n")
print(df.groupby("method")[[
    "valid_json", "issue_type", "urgency_level",
    "fraud_flag", "monetary_amount", "requested_actions",
    "sentiment", "overall"
]].mean().round(2))

print("\n=== Cost & Latency by Method ===\n")
print(df.groupby("method")[["latency", "input_tokens", "output_tokens"]].mean().round(2))

=== Field-Level Accuracy by Method ===

        valid_json  issue_type  urgency_level  fraud_flag  monetary_amount  \
method                                                                       
naive          0.0         0.0           0.00         0.0             0.00   
schema         0.0         0.0           0.00         0.0             0.00   
strict         1.0         1.0           0.67         0.0             0.67   

        requested_actions  sentiment  overall  
method                                         
naive                0.00        0.0     0.00  
schema               0.00        0.0     0.00  
strict               0.33        0.0     0.44  

=== Cost & Latency by Method ===

        latency  input_tokens  output_tokens
method                                      
naive      4.81         78.67         173.33
schema     2.48        115.67          88.00
strict     2.08        126.67          75.00


In [13]:
# Per-ticket breakdown — see where each method fails
print("=== Per Ticket Breakdown ===\n")
print(df[["ticket", "method", "valid_json", "issue_type", "urgency_level",
          "fraud_flag", "monetary_amount", "overall"]].to_string(index=False))

=== Per Ticket Breakdown ===

 ticket method  valid_json  issue_type  urgency_level  fraud_flag  monetary_amount  overall
      1  naive       False           0              0           0                0 0.000000
      1 schema       False           0              0           0                0 0.000000
      1 strict        True           1              1           0                0 0.333333
      2  naive       False           0              0           0                0 0.000000
      2 schema       False           0              0           0                0 0.000000
      2 strict        True           1              0           0                1 0.500000
      3  naive       False           0              0           0                0 0.000000
      3 schema       False           0              0           0                0 0.000000
      3 strict        True           1              1           0                1 0.500000


**Insight**

- Prompt constraints solve output formatting but not semantic accuracy




---



## Further Analysis


- Valid JSON rate went from 0% (naive, schema) to 100% (strict).
But overall field accuracy for strict is only 33–50%.
This means I have two separate failure layers


- The experiment proves that structured prompting is necessary
but not sufficient for production-grade extraction.


- fraud_flag is the highest-stakes field in a fintech triage
system. Getting it wrong has real consequences. Missed fraud means customer loss, liability, and regulatory risk.


## Next Steps


- Add explicit criteria for ambiguous fields

- Update scorer to handle int/float/string mismatches

- Add few-shot examples to the strict prompt

- Combine strict formatting rules + business rule definitions + one few-shot example

- 3 tickets is not enough to draw statistically meaningful conclusions



---

